# Week 5 Exercise — RAG Expert Knowledge Worker

A **question-answering assistant** over the Insurellm knowledge base using **RAG (Retrieval Augmented Generation)**.

- Load documents from `knowledge-base/` (employees, products, contracts, company)
- Chunk → embed (OpenAI) → store in Chroma
- Answer questions by retrieving relevant chunks and generating with the LLM
- **OpenAI only.** Run all cells; then use the Gradio chat.

**Note:** Run this notebook from the `week5/` directory (or ensure `knowledge-base/` is available).

In [ ]:
import os
import glob
from dotenv import load_dotenv
import gradio as gr

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


In [ ]:
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    print(f"OpenAI API key loaded (starts with {api_key[:8]}...)")
else:
    print("OPENAI_API_KEY not set")

MODEL = "gpt-4.1-mini"
DB_NAME = "week5_rag_db"

# Knowledge base path: works from week5/ or from week5/community-contributions/oluwatosinjegede/
ROOT = os.path.abspath(os.getcwd())
if "community-contributions" in ROOT:
    KB_PATH = os.path.normpath(os.path.join(ROOT, "..", "..", "knowledge-base"))
else:
    KB_PATH = os.path.normpath(os.path.join(ROOT, "knowledge-base"))
print(f"Knowledge base path: {KB_PATH}")
print(f"Exists: {os.path.isdir(KB_PATH)}")

In [ ]:
# Load all .md documents from knowledge-base subfolders; add doc_type metadata
documents = []
for folder in sorted(glob.glob(os.path.join(KB_PATH, "*"))):
    if not os.path.isdir(folder):
        continue
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(
        folder,
        glob="**/*.md",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"},
    )
    docs = loader.load()
    for doc in docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents from {KB_PATH}")

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
chunks = text_splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks")

In [ ]:
embeddings = OpenAIEmbeddings()

if os.path.exists(DB_NAME):
    Chroma(persist_directory=DB_NAME, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_NAME,
)
print(f"Vector store ready: {vectorstore._collection.count()} chunks")

In [ ]:
llm = ChatOpenAI(model=MODEL, temperature=0)
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

RAG_PROMPT = """You are an expert assistant for Insurellm, an insurance tech company.
Answer the question using only the following context. If the context does not contain the answer, say so briefly.
Be accurate and concise.

Context:
{context}

Question: {question}
Answer:"""

def format_docs(docs):
    return "\n\n---\n\n".join(d.page_content for d in docs)

def rag_invoke(question: str):
    docs = retriever.invoke(question)
    context = format_docs(docs)
    prompt = ChatPromptTemplate.from_messages([("human", RAG_PROMPT)])
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"context": context, "question": question})

print("RAG chain ready.")

In [ ]:
def chat(message, history):
    if not message or not message.strip():
        return "Ask a question about Insurellm (employees, products, contracts, company)."
    try:
        return rag_invoke(message.strip())
    except Exception as e:
        return f"Error: {e}"

In [ ]:
gr.ChatInterface(
    chat,
    title="Week 5 — RAG Expert Knowledge Worker",
    description="Ask questions about Insurellm (employees, products, contracts). Answers are grounded in the knowledge base.",
    type="messages",
    examples=["Who is Emily Carter?", "What is Claimllm?", "Which products does Insurellm offer?"],
).launch()